# Pré-processamento 2 — Arapiraca

Este notebook realiza:
1. Carregamento dos dados filtrados espacialmente
2. Cálculo dinâmico do tamanho do grid (área-alvo de 0,2 km² por célula)
3. Geração da grade espacial e visualização
4. Atribuição de cada ponto de crime à célula correspondente do grid

**Entrada:** `filtered_arapiraca_inside_polygon.csv`, `osm_arapiraca.geojson`

**Saída:** `arapiraca_with_grid_index.csv`, `arapiraca_grids.html`

## 1. Carregamento dos dados

In [1]:
# Carrega o CSV de Arapiraca filtrado espacialmente (dentro do polígono)

import pandas as pd

df_crimes = pd.read_csv(
    "./output/arapiraca/filtered_arapiraca_inside_polygon.csv",
    delimiter=";",
)
GRID_SIZE = 57 # from pre processing 1 

## 2. Visualização dos dados

In [2]:
display(df_crimes)
df_crimes.info()

,LONGITUDE,LATITUDE,ROUBO_GROUP,DATA_HORA,CIDADE_FATO
0,-36.690147,-9.735983,street_robbery,2022-01-01 03:00:00,Arapiraca
1,-36.655517,-9.785501,street_robbery,2022-01-01 20:00:00,Arapiraca
2,-36.591520,-9.762967,vehicle_robbery,2022-01-02 09:00:00,Arapiraca
3,-36.664139,-9.768197,street_robbery,2022-01-02 23:00:00,Arapiraca
4,-36.680604,-9.644294,street_robbery,2022-01-03 08:00:00,Arapiraca
...,...,...,...,...,...
17366,-36.662065,-9.764001,vehicle_robbery,2014-12-21 01:30:00,Arapiraca
17367,-36.666260,-9.753766,street_robbery,2014-12-22 14:30:00,Arapiraca
17368,-36.663400,-9.748675,street_robbery,2014-12-22 22:00:00,Arapiraca
17369,-36.648386,-9.721165,vehicle_robbery,2014-12-22 23:30:00,Arapiraca


<class 'pandas.DataFrame'>
RangeIndex: 17371 entries, 0 to 17370
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   LONGITUDE    17371 non-null  float64
 1   LATITUDE     17371 non-null  float64
 2   ROUBO_GROUP  17371 non-null  str    
 3   DATA_HORA    17371 non-null  str    
 4   CIDADE_FATO  17371 non-null  str    
dtypes: float64(2), str(3)
memory usage: 678.7 KB


## 3. Geração do grid e atribuição de células

- Cria uma grade regular N×N sobre o bounding box de Arapiraca
- Visualiza o grid em mapa interativo (folium)
- Para cada ponto de crime, identifica a célula do grid correspondente (point-in-polygon)
- Utiliza processamento paralelo (pandarallel) para acelerar a atribuição

In [3]:
# Geração do grid e atribuição de cada crime à sua célula
# generate_grid(): cria a grade N×N e visualiza em mapa
# find_polygon(): para cada ponto, encontra a célula que o contém
# Resultado: CSV com coluna 'cell' indicando o índice da célula

from shapely.geometry import Point
import shapely.geometry
import pandas as pd
import geopandas as gpd
import numpy as np
import folium
from pandarallel import pandarallel
import os

# Ensure at least 1 worker, handle case where os.cpu_count() returns None or 0
cpu_count = os.cpu_count() or 1
nb_workers = max(1, min(cpu_count, 20))
pandarallel.initialize(nb_workers=nb_workers, progress_bar=True)


def generate_grid(grid_size):
    mun = gpd.read_file("./output/arapiraca/osm_arapiraca.geojson")
    mun = mun.to_crs("EPSG:4326")
    gdf = mun[mun["name"] == "Arapiraca"].copy()

    bbox = gdf.total_bounds
    min_lon, min_lat, max_lon, max_lat = (bbox[2], bbox[1], bbox[0], bbox[3])

    lon = np.linspace(min_lon, max_lon, grid_size + 1)
    lat = np.linspace(min_lat, max_lat, grid_size + 1)

    latlons = []
    for i in range(len(lat) - 1):
        for k in range(len(lon) - 1):
            latlons.append((lat[k], lon[i], lat[k + 1], lon[i + 1]))

    m = folium.Map(
        location=((min_lat + max_lat) / 2, (min_lon + max_lon) / 2), zoom_start=11
    )
    for k in latlons:
        folium.Rectangle(
            [(k[0], k[1]), (k[2], k[3])], color="red", fill="pink", fill_opcity=0.5
        ).add_to(m)
    cgeo = (
        gdf.set_crs("epsg:4326")
        .sample(1)
        .pipe(lambda d: d.to_crs(d.estimate_utm_crs()))["geometry"]
        # .centroid.buffer(10000)
        .to_crs("epsg:4326")
        .__geo_interface__
    )

    geo_j = folium.GeoJson(data=cgeo)
    geo_j.add_to(m)
    m.save("./output/arapiraca/arapiraca_grids.html")

    return (min_lon, min_lat, max_lon, max_lat, latlons)


def find_polygon(row, df):
    for p_idx, elem in df.iterrows():
        if elem["polygon"].contains(Point(row["LONGITUDE"], row["LATITUDE"])):
            polygon_num = p_idx
            break
    return pd.Series({"cell": polygon_num})


# generate grid boxes
min_lon, min_lat, max_lon, max_lat, grid_bboxes = generate_grid(grid_size=GRID_SIZE)

# turn boxes into polygons and that list into a dataframe
polygon_list = []
for elem in grid_bboxes:
    # elem is (min_lat, min_lon, max_lat, max_lon)
    polygon = shapely.geometry.box(elem[1], elem[0], elem[3], elem[2], ccw=True)
    polygon_list.append(polygon)
df_poly = pd.DataFrame(polygon_list, columns=["polygon"])


crimes_list = [
    "street_robbery",
    "vehicle_robbery",
    "public_transport_robbery",
    "commercial_robbery",
    "residential_robbery",
]
# generate file for each crime
print("Finding the grid cell for each point...")
df_final = df_crimes.merge(
    df_crimes.parallel_apply(find_polygon, df=df_poly, axis=1),
    left_index=True,
    right_index=True,
)

df_final.to_csv(f"./output/arapiraca/arapiraca_with_grid_index.csv", sep=";")
display(df_final.shape)

INFO: Pandarallel will run on 20 workers.
INFO: Pandarallel will use Memory file system to transfer data between the main process and workers.
Finding the grid cell for each point...


(17371, 6)